In [3]:
"""
qwen2.5vl:3b
"""
import ollama
import base64
import json
import os

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

def chat_once(user_text, image_path=None, history=None):
    if history is None:
        history = []
    
    if image_path:
        message = {
            "role": "user",
            "content": user_text,
            "images": [encode_image(image_path)]
        }
    else:
        message = {
            "role": "user",
            "content": user_text
        }
    
    history.append(message)
    
    response = ollama.chat(
        model="qwen2.5vl:3b",
        messages=history,
        options={"num_ctx": 8192}
    )
    
    return response["message"]["content"]


image_path_1 = "Scene/Scene1.png"
target_question_1 = "what is to the left of the sofa and what is the colour of it?"

image_path_2 = "Scene/Scene2.png"
target_question_2 = "what is to the right of the bed and what is the colour of it?"

image_path_3 = "Scene/Scene3.png"
target_question_3 = "How many people are standing to the left of the man wearing gray T-shirt in the middle? What are they wearing?"

conditions = {
    "relative": f"From your perspective looking at the scene, {target_question_3}",
    "intrinsic": f"From the man in the gray T-shirt's own perspective, {target_question_3}", #change the prompt when switching the task
    "hearer_180": f"I am standing opposite you. From my perspective, {target_question_3}"
}

results = {}

for condition_name, prompt in conditions.items():
    answer = chat_once(prompt, image_path=image_path_3, history=[])
    results[condition_name] = answer
    print(f"[{condition_name}]")
    print(answer)
    print()

file_path = "task1_results_qwen.json"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

In [ ]:
t1 = chat("I'm the man wearing a blue T-shirt, standing to the right of the bed. Can you see me?",image_path=image_path_2)
print("T1:", t1)

t2 = chat("There is a nightstand to the right of the bed, which nightstand is it and what's the colour of the nightstand?",image_path=image_path_2)
print("T2:", t2)

t3 = chat("There is a table to the left of the bed, which table is it and what's the colour of the table?",image_path=image_path_2)
print("T3:", t3)

t4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path_2)
print("T4:", t4)

In [ ]:
t1 = chat("I'm the man wearing a blue T-shirt, standing to the left of the man wearing gray T-shirt in the middle. Can you see me?",image_path=image_path_3)
print("T1:", t1)

t2 = chat("There are two people standing closest to the right of the man in the gray T-shirt in the middle. What are they wearing?",image_path=image_path_3)
print("T2:", t2)

t3 = chat("There is a single people standing closest to the left of the man wearing a gray T-shirt in the middle. What is that person wearing?",image_path=image_path_3)
print("T3:", t3)

t4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path_3)
print("T4:", t4)

In [2]:
import ollama
import base64
import json
import os
def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

conversation_history = []

def chat(user_text, image_path=None):
    if image_path:
        message = {
            "role": "user",
            "content": user_text,
            "images": [encode_image(image_path)]
        }
    else:
        message = {
            "role": "user", 
            "content": user_text
        }
    
    conversation_history.append(message)
    
    response = ollama.chat(
        model="qwen2.5vl:3b",
        messages=conversation_history,
        options={"num_ctx": 8192}
    )
    
    assistant_message = {
        "role": "assistant",
        "content": response["message"]["content"]
    }
    conversation_history.append(assistant_message)
    
    return response["message"]["content"]

image_path = "Scene/Scene2.png"

turn1 = chat("I'm the man wearing a blue T-shirt, standing to the right of the bed. Can you see me?", image_path=image_path)
print("T1:", turn1)

turn2 = chat("There is a nightstand to the right of the bed, which nightstand is it and what's the colour of the nightstand?",image_path=image_path)
print("T2:", turn2)

turn3 = chat("There is a table to the left of the bed, which table is it and what's the colour of the table?",image_path=image_path)
print("T3:", turn3)

turn4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path)
print("T4:", turn4)

results = {
    "scene": image_path,
    "turn1_priming": turn1,
    "turn2_probe": turn2,
    "turn3_probe": turn3,
    "turn4_free_description": turn4
}

file_path = "task2_results_qwen.json"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

T1: Yes, you are visible in the image. You are standing to the right of the bed, wearing a blue T-shirt.
T2: The nightstand to the right of the bed is a small, rectangular object with a light grey color.
T3: The table to the left of the bed is a rectangular table with a light grey color.
T4: The man in the red T-shirt is standing to the right of the bed.


In [ ]:
"""
Gemini-3.1-flash-lite
"""
from google import genai
from PIL import Image
import json
import os

client = genai.Client(api_key)

chat = client.chats.create(model="gemini-3.1-flash-lite")

image_path_1 = "Scene/Scene1.png"
image_1 = Image.open(image_path_1)
target_question_1 = "what is to the left of the sofa and what is the colour of it?"

image_path_2 = "Scene/Scene2.png"
image_2 = Image.open(image_path_2)
target_question_2 = "what is to the right of the bed and what is the colour of it?"

image_path_3 = "Scene/Scene3.png"
image_3 = Image.open(image_path_3)
target_question_3 = "How many people are standing to the left of the man wearing gray T-shirt in the middle? What are they wearing?"

conditions = {
    "relative": f"From your perspective looking at the scene, {target_question_3}",
    "intrinsic": f"From the man in the gray T-shirt's own perspective, {target_question_3}",
    "hearer_180": f"I am standing opposite you. From my perspective, {target_question_3}"
}

results = {}

for condition_name, prompt in conditions.items():
    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=[prompt, image_3]
    )
    results[condition_name] = response.text
    print(f"[{condition_name}]")
    print(response.text)
    print()

file_path = "task1_results_Gemini.json"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

In [ ]:
response1 = chat.send_message(["There is a TV in front of the sofa.",image_2])
print("T1:", response1.text)

response2 = chat.send_message(["There is a chair to the left of the sofa, which chair is it and what's the colour of the chair?",image_2])
print("T2:", response2.text)

response3 = chat.send_message(["There is a table to the right of the sofa, which table is it and what's the colour of the table?",image_2])
print("T3:", response3.text)

response4 = chat.send_message(["Where is the wardrobe? Please describe its location.",image_2])
print("T4:", response4.text)

In [ ]:
t1 = chat("I'm the man wearing a blue T-shirt, standing to the right of the bed. Can you see me?",image_path=image_path_2)
print("T1:", t1)

t2 = chat("There is a nightstand to the right of the bed, which nightstand is it and what's the colour of the nightstand?",image_path=image_path_2)
print("T2:", t2)

t3 = chat("There is a table to the left of the bed, which table is it and what's the colour of the table?",image_path=image_path_2)
print("T3:", t3)

t4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path_2)
print("T4:", t4)

In [ ]:
t1 = chat("I'm the man wearing a blue T-shirt, standing to the left of the man wearing gray T-shirt in the middle. Can you see me?",image_path=image_path_3)
print("T1:", t1)

t2 = chat("There are two people standing closest to the right of the man in the gray T-shirt in the middle. What are they wearing?",image_path=image_path_3)
print("T2:", t2)

t3 = chat("There is a single people standing closest to the left of the man wearing a gray T-shirt in the middle. What is that person wearing?",image_path=image_path_3)
print("T3:", t3)

t4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path_3)
print("T4:", t4)

In [ ]:
from google import genai
from PIL import Image
import json
import os

client = genai.Client(api_key)

chat = client.chats.create(model="gemini-3.1-flash-lite")

image_path_1 = "Scene/Scene1.png"
image_1 = Image.open(image_path_1)
target_question_1 = "what is to the left of the sofa and what is the colour of it?"

image_path_2 = "Scene/Scene2.png"
image_2 = Image.open(image_path_2)
target_question_2 = "what is to the right of the bed and what is the colour of it?"

image_path_3 = "Scene/Scene3.png"
image_3 = Image.open(image_path_3)

target_question_3 = "How many people are standing to the left of the man wearing gray T-shirt in the middle? What are they wearing?"

response1 = chat.send_message(["I'm the man wearing a blue T-shirt, standing to the right of the bed. Can you see me?",image_2])
print("T1:", response1.text)

response2 = chat.send_message(["There is a nightstand to the right of the bed, which nightstand is it and what's the colour of the nightstand?",image_2])
print("T2:", response2.text)

response3 = chat.send_message(["There is a table to the left of the bed, which table is it and what's the colour of the table?",image_2])
print("T3:", response3.text)

response4 = chat.send_message(["Where is the man in the red T-shirt? Please describe its location.",image_2])
print("T4:", response4.text)

results = {
    "scene": image_path_2,
    "turn1_priming": response1.text,
    "turn2_probe": response2.text,
    "turn3_probe": response3.text,
    "turn4_free_description": response4.text
}

file_path = "task2_results_Gemini.json"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

T1: I can see a person in a blue shirt in the image, but they are positioned **behind** the bed, not to the right of it. The person standing to the right of the bed is wearing a red shirt.
T2: The object to the right of the bed is a wooden cabinet or chest. It is light brown or tan in color.
T3: The object to the left of the bed is a dining or rectangular table. It has a wood-grain texture and is brown in color.
T4: The man in the red T-shirt is standing on the right side of the room, positioned behind the wooden cabinet and to the right of the bed.


In [ ]:
"""
Kimi 2.6
"""
from openai import OpenAI
from PIL import Image
import json
import os
import base64

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

client = OpenAI(
    api_key,
    base_url="https://api.moonshot.cn/v1"
)

image_path_1 = "Scene/Scene1.png"
target_question_1 = "what is to the left of the sofa and what is the colour of it?"

image_path_2 = "Scene/Scene2.png"
target_question_2 = "what is to the right of the bed and what is the colour of it?"

image_path_3 = "Scene/Scene3.png"
target_question_3 = "How many people are standing to the left of the man wearing gray T-shirt in the middle? What are they wearing?"

conditions = {
    "relative": f"From your perspective looking at the scene, {target_question_1}",
    "intrinsic": f"From the sofa's own perspective, {target_question_1}",
    "hearer_180": f"I am standing opposite you. From my perspective, {target_question_1}"
}

results = {}

for condition_name, prompt in conditions.items():
    response = client.chat.completions.create(
        model="kimi-k2.6",
        messages=[{
            "role": "user",
            "content": [
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": {
                    "url": f"data:image/png;base64,{encode_image(image_path_1)}"
                }}
            ]
        }]
    )
    results[condition_name] = response.choices[0].message.content
    print(f"[{condition_name}]")
    print(results[condition_name])
    print()

file_path = "task1_results_Kimi.json"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

In [ ]:
from openai import OpenAI
from PIL import Image
import json
import os
import base64

def encode_image(image_path):
    with open(image_path, "rb") as f:
        return base64.b64encode(f.read()).decode("utf-8")

client = OpenAI(
    api_key=,
    base_url="https://api.moonshot.cn/v1"
)

image_path_1 = "Scene/Scene1.png"
image_path_2 = "Scene/Scene2.png"
image_path_3 = "Scene/Scene3.png"

messages = []

def chat(user_text, image_path=None):
    if image_path:
        content = [
            {"type": "text", "text": user_text},
            {"type": "image_url", "image_url": {
                "url": f"data:image/png;base64,{encode_image(image_path)}"
            }}
        ]
    else:
        content = user_text

    messages.append({"role": "user", "content": content})

    response = client.chat.completions.create(
        model="kimi-k2.6",
        messages=messages
    )

    reply = response.choices[0].message.content
    messages.append({"role": "assistant", "content": reply})
    return reply

t1 = chat("I'm the man wearing a blue T-shirt, standing to the right of the bed. Can you see me?",image_path=image_path_2)
print("T1:", t1)

t2 = chat("There is a nightstand to the right of the bed, which nightstand is it and what's the colour of the nightstand?",image_path=image_path_2)
print("T2:", t2)

t3 = chat("There is a table to the left of the bed, which table is it and what's the colour of the table?",image_path=image_path_2)
print("T3:", t3)

t4 = chat("Where is the man in the red T-shirt? Please describe its location.",image_path=image_path_2)
print("T4:", t4)

results = {
    "scene": image_path_2,
    "turn1_priming": t1,
    "turn2_probe": t2,
    "turn3_probe": t3,
    "turn4_free_description": t4
}

file_path = "task2_results_Kimi.json"
if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        try:
            old = json.load(f)
            if not isinstance(old, list):
                old = [old]
        except:
            old = []
else:
    old = []

old.append(results)

with open(file_path, "w", encoding="utf-8") as f:
    json.dump(old, f, ensure_ascii=False, indent=2)

T1: I can see the figure you're referring to, but I need to clarify the positions. In the image, the **man wearing the blue T-shirt is standing near the back wall**, behind the small white cabinet—not to the right of the bed.

The figure **to the right of the bed** (near the wooden cabinet on the right side) is actually wearing a **red shirt** and dark pants.

Could you be thinking of the figure in the red shirt, or are you referring to the blue-shirted figure near the back of the room?
T2: Based on the image, the nightstand to the **right** of the bed is the **wooden cabinet** positioned next to the man in the red shirt. Its color is **light brown** (a beige or natural wood tone).

This is different from the **white** nightstand/cabinet that sits behind the bed, near the back wall by the man in the blue shirt.
T3: The table to the **left** of the bed is the wooden table with the rectangular top. Its colour is **brown** (with lighter wooden legs).
T4: The man in the red T-shirt is stan